# INSTRUCCIONES

Para preparar esta demo:
1. Primero debo ejecutar el cuaderno al completo
2. Cuando esté ejecutándose el `demo.launch(Share=True)` debo capturar el enlace de gradio generado: https://[XXX].gradio.live/
3. Pegar el enlace generado en https://huggingface.co/spaces/visumania2/DeepSexist-App/tree/main en el archivo `index.html`
4. El espace ya estará disponible con la demo de la prueba

# 1. Instalamos las librerías necesarias y conexión con Drive

In [1]:
!pip install -q gradio transformers torch decord opencv-python moviepy bitsandbytes accelerate peft "torchao>=0.16.0" av

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.6/13.6 MB 124.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 57.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 113.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.8/35.8 MB 70.4 MB/s eta 0:00:00


In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


# 2. Importaciones y Configuración del Ensemble

In [3]:
import gradio as gr
from transformers import pipeline, BitsAndBytesConfig
import torch
import decord
import cv2
import moviepy.editor as mp
import numpy as np
import gc

/usr/local/lib/python3.13/dist-packages/moviepy/config_defaults.py:47: SyntaxWarning: invalid escape sequence '\P'
  IMAGEMAGICK_BINARY = r"C:\Program Files\ImageMagick-6.8.8-Q16\magick.exe"
/usr/local/lib/python3.13/dist-packages/moviepy/video/io/ffmpeg_reader.py:294: SyntaxWarning: invalid escape sequence '\d'
  lines_video = [l for l in lines if ' Video: ' in l and re.search('\d+x\d+', l)]
/usr/local/lib/python3.13/dist-packages/moviepy/video/io/ffmpeg_reader.py:367: SyntaxWarning: invalid escape sequence '\d'
  rotation_lines = [l for l in lines if 'rotate          :' in l and re.search('\d+$', l)]
/usr/local/lib/python3.13/dist-packages/moviepy/video/io/ffmpeg_reader.py:370: SyntaxWarning: invalid escape sequence '\d'
  match = re.search('\d+$', rotation_line)
  if event.key is 'enter':



In [4]:
# Definición de pesos de los modelos en el ensemble
PESO_TEXTO = 0.35
PESO_IMAGEN = 0.55
PESO_VIDEO = 0.10

In [5]:
# Función vital para limpiar la memoria VRAM de la GPU y evitar que Colab explote
def liberar_memoria():
    gc.collect()
    torch.cuda.empty_cache()

# 3. Lógica de Procesamiento Secuencial

In [6]:
from transformers import pipeline, BitsAndBytesConfig # <--- Importamos el configurador

def procesar_texto(texto):
    if not texto.strip(): return 0.0, "Texto vacío"

    # 1. Empaquetamos la configuración de 4-bits de forma oficial
    quant_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_compute_dtype=torch.float16 # Esto ayuda a que vaya más rápido en la T4 de Colab
    )

    # 2. Cargar Mistral desde Drive con la configuración correcta
    RUTA_MISTRAL = "/content/drive/MyDrive/dataset/EXIST 2025 Videos Dataset/training/Research/LLMs/Mistral_7B_QLoRA/checkpoint-565"

    clasificador_txt = pipeline(
        "text-classification",
        model=RUTA_MISTRAL,
        device_map="auto",
        model_kwargs={"quantization_config": quant_config} # <--- Usamos el paquete aquí
    )

    # 3. Inferencia
    prediccion = clasificador_txt(texto)[0]
    etiqueta = str(prediccion['label'])
    score = prediccion['score']

    # 4. Formateo visual
    if etiqueta in ['LABEL_1', '1', 'YES', 'yes', 'sexist']:
        prob_sexista = score
        texto_resultado = f"⚠️ CLASE: SEXISTA (Probabilidad de Sexismo: {prob_sexista:.2%})"
    else:
        prob_sexista = 1.0 - score
        texto_resultado = f"✅ CLASE: NO SEXISTA (Probabilidad de Sexismo: {prob_sexista:.2%})"

    # 5. Borrar modelo y limpiar RAM
    del clasificador_txt
    liberar_memoria()

    return prob_sexista, texto_resultado

In [7]:
def procesar_imagen(ruta_video):
    if not ruta_video: return 0.0, "Vídeo no válido"

    # 1. Extraer 4 frames equiespaciados con CV2
    cap = cv2.VideoCapture(ruta_video)
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    frames_a_extraer = [int(total_frames * i / 4) for i in range(4)]

    frames_extraidos = []
    for frame_idx in frames_a_extraer:
        cap.set(cv2.CAP_PROP_POS_FRAMES, frame_idx)
        ret, frame = cap.read()
        if ret:
            # CV2 lee en BGR, Hugging Face necesita RGB
            frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            frames_extraidos.append(frame_rgb)
    cap.release()

    # 2. Cargar ConvNeXt desde Drive
    RUTA_CONVNEXT = "/content/drive/MyDrive/dataset/EXIST 2025 Videos Dataset/training/Research/Vision/CONVNEXT_Frames_4/modelo_final"
    clasificador_img = pipeline("image-classification", model=RUTA_CONVNEXT, device=0)

    # 3. Mean-Pooling con lógica de etiquetas para los 4 frames
    probs_sexista = []
    for frame in frames_extraidos:
        prediccion = clasificador_img(frame)[0]
        etiqueta = str(prediccion['label'])
        score = prediccion['score']

        # Ajustamos la probabilidad según la clase que prediga
        if etiqueta in ['LABEL_1', '1', 'YES', 'yes', 'sexist']:
            probs_sexista.append(score)
        else:
            probs_sexista.append(1.0 - score)

    # Calculamos la media de las 4 probabilidades
    prob_sexista_media = sum(probs_sexista) / len(probs_sexista) if probs_sexista else 0.0

    # 4. Formateo visual
    if prob_sexista_media > 0.5:
        texto_resultado = f"⚠️ CLASE: SEXISTA (Probabilidad de Sexismo: {prob_sexista_media:.2%})"
    else:
        texto_resultado = f"✅ CLASE: NO SEXISTA (Probabilidad de Sexismo: {prob_sexista_media:.2%})"

    # 5. Borrar modelo y limpiar RAM
    del clasificador_img
    liberar_memoria()

    return prob_sexista_media, texto_resultado

In [8]:
def procesar_video(ruta_video):
    if not ruta_video: return 0.0, "Vídeo no válido"

    # 1. Cargar TimeSFormer desde Drive
    RUTA_TIMESFORMER = "/content/drive/MyDrive/dataset/EXIST 2025 Videos Dataset/training/Research/Vision_Temporal/TIMESFORMER_FineTuned_experimentacion/modelo_final"
    clasificador_vid = pipeline("video-classification", model=RUTA_TIMESFORMER, device=0)

    # 2. Inferencia
    prediccion = clasificador_vid(ruta_video)[0]
    etiqueta = str(prediccion['label'])
    score = prediccion['score']

    # 3. Lógica de etiquetas y formateo visual
    if etiqueta in ['LABEL_1', '1', 'YES', 'yes', 'sexist']:
        prob_sexista = score
        texto_resultado = f"⚠️ CLASE: SEXISTA (Probabilidad de Sexismo: {prob_sexista:.2%})"
    else:
        prob_sexista = 1.0 - score
        texto_resultado = f"✅ CLASE: NO SEXISTA (Probabilidad de Sexismo: {prob_sexista:.2%})"

    # 4. Borrar modelo y limpiar RAM
    del clasificador_vid
    liberar_memoria()

    return prob_sexista, texto_resultado

In [13]:
def procesar_video_ensemble(ruta_video):
    if not ruta_video: return "Error", "Falta vídeo", "Falta vídeo", "Falta vídeo", "Por favor, sube un vídeo."

    # --- FASE 1: TEXTO (Whisper + Mistral) ---
    # A) Extraemos audio
    ruta_audio = "audio_temp.wav"
    clip = mp.VideoFileClip(ruta_video)
    clip.audio.write_audiofile(ruta_audio, logger=None)

    # B) Transcribimos con Whisper (con troceado automático para vídeos largos)
    transcriptor = pipeline(
        "automatic-speech-recognition",
        model="openai/whisper-small",
        device=0,
        chunk_length_s=30  # <--- En caso de que el video tenga más de 30 segundos, lo trocea
    )
    transcripcion = transcriptor(ruta_audio)["text"]

    # C) Limpiamos Whisper de la memoria y pasamos a Mistral
    del transcriptor
    liberar_memoria()

    prob_t, res_mistral = procesar_texto(transcripcion)

    # --- FASE 2: IMAGEN (ConvNeXt + Mean Pooling) ---
    prob_i, res_convnext = procesar_imagen(ruta_video)

    # --- FASE 3: VÍDEO (TimeSFormer) ---
    prob_v, res_timesformer = procesar_video(ruta_video)

    # --- FASE 4: ENSEMBLE (LATE FUSION) ---
    probabilidad_final = (prob_t * PESO_TEXTO) + (prob_i * PESO_IMAGEN) + (prob_v * PESO_VIDEO)

    if probabilidad_final > 0.5:
        veredicto_final = f"⚠️ CLASE: SEXISTA (Probabilidad Ensemble: {probabilidad_final:.2%})"
    else:
        veredicto_final = f"✅ CLASE: NO SEXISTA (Probabilidad Ensemble: {probabilidad_final:.2%})"

    return transcripcion, res_mistral, res_convnext, res_timesformer, veredicto_final

# 4. Interfaz y Lanzamiento

In [ ]:
# 4. Interfaz y Lanzamiento
with gr.Blocks(theme=gr.themes.Base()) as demo:
    gr.Markdown("<h1 style='text-align: center;'>🧠 Detección Multimodal de Sexismo en Vídeo (Ensemble)</h1>")
    gr.Markdown("<p style='text-align: center; font-size: 1.1em; color: #555;'>Sube un vídeo. El sistema transcribirá el audio para <b>Mistral</b>, extraerá 4 frames para <b>ConvNeXt</b>, y analizará el movimiento con <b>TimeSFormer</b> para dar un veredicto conjunto (<i>Late Fusion</i>).</p>")

    with gr.Row():
        # Columna izquierda para la entrada del vídeo
        with gr.Column(scale=1):
            vid_in = gr.Video(label="Sube tu vídeo a analizar")
            btn_vid = gr.Button("Ejecutar Pipeline Multimodal", variant="primary", size="lg")
            out_trans = gr.Textbox(label="0. Transcripción generada (Whisper)", lines=3)

        # Columna derecha para los resultados
        with gr.Column(scale=1):
            out_t = gr.Textbox(label="1. Vía Texto (Mistral - 35%)")
            out_i = gr.Textbox(label="2. Vía Imagen (ConvNeXt - 55%)")
            out_v = gr.Textbox(label="3. Vía Vídeo (TimeSFormer - 10%)")
            out_final = gr.Textbox(label="🏆 VEREDICTO FINAL (ENSEMBLE)", lines=2)

    # Conectamos el botón con la función principal
    btn_vid.click(
        procesar_video_ensemble,
        inputs=vid_in,
        outputs=[out_trans, out_t, out_i, out_v, out_final]
    )

if __name__ == "__main__":
    demo.launch(share=True, debug=True)

  with gr.Blocks(theme=gr.themes.Base()) as demo:



Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://cf4a6a855ccc4e70e8.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Loading weights:   0%|          | 0/479 [00:00<?, ?it/s]

[transformers] Using `chunk_length_s` is very experimental with seq2seq models. The results will not necessarily be entirely accurate and will have caveats. More information: https://github.com/huggingface/transformers/pull/20104. Ignore this warning with pipeline(..., ignore_warning=True). To use Whisper for long-form transcription, use rather the model's `generate` method directly as the model relies on it's own chunking mechanism (cf. Whisper original paper, section 3.8. Long-form Transcription).


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

...

[transformers] MistralForSequenceClassification LOAD REPORT from: mistralai/Mistral-7B-Instruct-v0.3
Key            | Status     | 
---------------+------------+-
lm_head.weight | UNEXPECTED | 
score.weight   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Loading weights:   0%|          | 0/448 [00:00<?, ?it/s]

[transformers] MistralForSequenceClassification LOAD REPORT from: /content/drive/MyDrive/dataset/EXIST 2025 Videos Dataset/training/Research/LLMs/Mistral_7B_QLoRA/checkpoint-565
Key                                  | Status     | 
-------------------------------------+------------+-
score.weight                         | UNEXPECTED | 
score.modules_to_save.default.weight | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Loading weights:   0%|          | 0/182 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/249 [00:00<?, ?it/s]